<a href="https://colab.research.google.com/github/obaidah3/rag-ecommerce-chatbot/blob/main/04_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4. Q&A RAG Pipeline
Retrieval-Augmented Generation over the Bitext `instruction` (question) / `response` (answer) pairs.

**Stack (all free, all local except the LLM call):**
- Embeddings: `sentence-transformers/all-MiniLM-L6-v2`
- Vector store: **FAISS**, local — chosen over cloud Qdrant to avoid external account setup/latency under a 1-day deadline. (Swapping to Qdrant later is a ~10 line change if needed.)
- LLM: Groq API, `gpt-oss-20b` (or `gpt-oss-120b` for higher quality if you have quota) — free tier, very fast inference, good for live demo/assessment.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/chatbot_project"
MODELS_DIR = f"{PROJECT_DIR}/models"
os.makedirs(MODELS_DIR, exist_ok=True)
print("Models will be saved to:", MODELS_DIR)

Mounted at /content/drive
Models will be saved to: /content/drive/MyDrive/chatbot_project/models


In [ ]:
!pip install -q sentence-transformers faiss-cpu groq datasets pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 9.4 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("Key loaded:", bool(os.environ["GROQ_API_KEY"]))
print("Prefix:", os.environ["GROQ_API_KEY"][:4])
print("Length:", len(os.environ["GROQ_API_KEY"]))

Key loaded: True
Prefix: 3IrZ
Length: 100


In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = ds["train"].to_pandas()

# De-duplicate near-identical instructions to keep the index lean and retrieval clean
df = df.drop_duplicates(subset=["instruction"]).reset_index(drop=True)
print(df.shape)


(24635, 5)


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# We embed the CUSTOMER QUESTION (instruction) since that's what we match incoming
# queries against; the paired RESPONSE is what we inject as grounding context.
corpus_texts = df["instruction"].tolist()
embeddings = embedder.encode(corpus_texts, batch_size=128, show_progress_bar=True, normalize_embeddings=True)
embeddings = np.asarray(embeddings, dtype="float32")
embeddings.shape


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/193 [00:00<?, ?it/s]

(24635, 384)

In [ ]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)   # cosine similarity via normalized inner product
index.add(embeddings)
faiss.write_index(index, f"{MODELS_DIR}/rag_index.faiss")

df[["instruction", "response", "intent", "category"]].to_parquet(f"{MODELS_DIR}/rag_corpus.parquet")
print("index size:", index.ntotal)


index size: 24635


In [ ]:
def retrieve(query, k=3):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        results.append({
            "instruction": df.iloc[idx]["instruction"],
            "response": df.iloc[idx]["response"],
            "score": float(score)
        })
    return results

retrieve("where is my order")


[{'instruction': 'where do I order something?',
  'response': "I'm delighted to assist you in finding the right place to order our products! To place an order, you can visit our website at {{Website URL}}. Our website is designed to provide you with a user-friendly and seamless ordering experience. Simply search for the items you wish to purchase, add them to your cart, and proceed to the checkout page to complete your order. Our website also offers secure payment options to ensure a smooth and protected transaction. If you have any questions or need further assistance during the ordering process, our customer support team is always available to help you. Happy shopping!",
  'score': 0.7598791122436523},
 {'instruction': 'where to see when my order is gonna arrive',
  'response': 'We understand your curiosity about tracking the status of your order and when it will arrive. To provide you with the most accurate information, could you please share the {{Tracking Number}} or {{Order Numbe

In [ ]:
from groq import Groq

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

SYSTEM_PROMPT = '''You are a helpful, professional customer support assistant for an online retailer.
Answer the customer's question using ONLY the information in the retrieved support responses below.
If the customer sounds frustrated ({sentiment}), acknowledge that before answering.
If the retrieved context does not cover the question, say so honestly and offer to escalate to a human agent rather than guessing.'''

def generate_answer(user_message, sentiment="neutral", k=3, model="openai/gpt-oss-20b"):
    chunks = retrieve(user_message, k=k)
    context = "\n\n".join([f"- Q: {c['instruction']}\n  A: {c['response']}" for c in chunks])

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT.format(sentiment=sentiment)},
        {"role": "user", "content": f"Context (retrieved past support responses):\n{context}\n\nCustomer question: \"{user_message}\""}
    ]
    resp = groq_client.chat.completions.create(model=model, messages=messages, temperature=0.3)
    return resp.choices[0].message.content, chunks


In [ ]:
answer, sources = generate_answer("I never received my refund for order 88213, this is ridiculous!", sentiment="negative")
print(answer)
print("\n--- sources ---")
for s in sources:
    print(s["instruction"], "| score:", round(s["score"],3))


AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

In [ ]:
import requests

headers = {"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"}
r = requests.get("https://api.groq.com/openai/v1/models", headers=headers)
print(r.status_code)
print(r.text[:300])

401
{"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"invalid_api_key"}}



In [ ]:
print(len(os.environ["GROQ_API_KEY"]))
print(os.environ["GROQ_API_KEY"][:6], "...", os.environ["GROQ_API_KEY"][-4:])

50
33IrY1 ... Q3x6


In [ ]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("Key loaded:", bool(os.environ["GROQ_API_KEY"]))

Key loaded: True


In [ ]:
from google.colab import userdata

key = userdata.get("GROQ_API_KEY")

print("Exists:", key is not None)
print("Length:", len(key) if key else 0)
print("Prefix:", key[:5] if key else None)

Exists: True
Length: 54
Prefix: gsk_3
